# Step 6: Performance Metrics & Strategy Analytics Framework

This notebook implements and demonstrates the quantitative evaluation framework built to analyze backtest simulation results.

### Objective
Load backtested histories and trade books for both **Momentum** and **Mean Reversion** strategies, calculate comprehensive return, risk, trade, and distribution statistics, generate drawdowns logs, scorecards, and compile all 20 required charts.

In [ ]:
import os
import sys

# Insert project source root to system path for importing local modules
sys.path.append(os.path.abspath('../'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.metrics import calculate_return_metrics, calculate_risk_adjusted_ratios, calculate_distribution_metrics
from src.risk import calculate_volatility, calculate_maximum_drawdown
from src.analytics import calculate_trade_statistics, calculate_benchmark_metrics, calculate_rolling_metrics, analyze_drawdowns, generate_scorecard, plot_performance_dashboard
from src.validators import validate_analytics_inputs

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Setup complete. Modules imported successfully.")

## 1. Load Price Data & Backtest Results

We load the clean benchmark prices (Nifty 50) and backtest output files.

In [ ]:
benchmark_df = pd.read_parquet("../data/processed/nifty50_clean.parquet")

mom_portfolio = pd.read_parquet("../data/processed/momentum_portfolio_history.parquet")
mom_trades = pd.read_csv("../data/processed/momentum_trade_book.csv")
mom_trades["Entry Date"] = pd.to_datetime(mom_trades["Entry Date"])
mom_trades["Exit Date"] = pd.to_datetime(mom_trades["Exit Date"])

mr_portfolio = pd.read_parquet("../data/processed/mean_reversion_portfolio_history.parquet")
mr_trades = pd.read_csv("../data/processed/mean_reversion_trade_book.csv")
mr_trades["Entry Date"] = pd.to_datetime(mr_trades["Entry Date"])
mr_trades["Exit Date"] = pd.to_datetime(mr_trades["Exit Date"])

print(f"Momentum History days: {len(mom_portfolio)}, Trades count: {len(mom_trades)}")
print(f"Mean Reversion History days: {len(mr_portfolio)}, Trades count: {len(mr_trades)}")

## 2. Input Validations

We run validation checks on the loaded dataframes to confirm index consistency and column types.

In [ ]:
validate_analytics_inputs(mom_portfolio, mom_trades)
validate_analytics_inputs(mr_portfolio, mr_trades)
print("Analytics inputs successfully validated!")

## 3. Momentum Strategy Evaluation

We run the metrics and scorecard generator for the Momentum strategy.

In [ ]:
aligned_bench = benchmark_df.reindex(mom_portfolio.index).ffill()
bench_returns = aligned_bench["Close"].pct_change().fillna(0.0)

mom_ret = calculate_return_metrics(mom_portfolio["Portfolio_Value"], mom_portfolio["Daily_Return"])
mom_bench = calculate_benchmark_metrics(mom_portfolio["Daily_Return"], bench_returns)
mom_risk_ratios = calculate_risk_adjusted_ratios(
    mom_portfolio["Daily_Return"], 
    mom_portfolio["Portfolio_Value"],
    benchmark_returns=bench_returns,
    beta=mom_bench.get("Beta", 1.0)
)
mom_trade_stats = calculate_trade_statistics(mom_trades)
mom_max_dd, _, _, _ = calculate_maximum_drawdown(mom_portfolio["Portfolio_Value"])

# Combine all metrics
mom_summary = {**mom_ret, **mom_bench, **mom_risk_ratios, **mom_trade_stats, "Max_Drawdown": mom_max_dd * 100.0}

print("=== Momentum Scorecard ===")
mom_scorecard = generate_scorecard(mom_summary)
display(mom_scorecard)

### Inspect Top 5 Worst Momentum Drawdown Events

We compile the drawdown event log and view the worst events.

In [ ]:
mom_dd_events = analyze_drawdowns(mom_portfolio["Portfolio_Value"])
display(mom_dd_events.head(5))

## 4. Mean Reversion Strategy Evaluation

We run the metrics and scorecard generator for the Mean Reversion strategy.

In [ ]:
aligned_bench_mr = benchmark_df.reindex(mr_portfolio.index).ffill()
bench_returns_mr = aligned_bench_mr["Close"].pct_change().fillna(0.0)

mr_ret = calculate_return_metrics(mr_portfolio["Portfolio_Value"], mr_portfolio["Daily_Return"])
mr_bench = calculate_benchmark_metrics(mr_portfolio["Daily_Return"], bench_returns_mr)
mr_risk_ratios = calculate_risk_adjusted_ratios(
    mr_portfolio["Daily_Return"], 
    mr_portfolio["Portfolio_Value"],
    benchmark_returns=bench_returns_mr,
    beta=mr_bench.get("Beta", 1.0)
)
mr_trade_stats = calculate_trade_statistics(mr_trades)
mr_max_dd, _, _, _ = calculate_maximum_drawdown(mr_portfolio["Portfolio_Value"])

mr_summary = {**mr_ret, **mr_bench, **mr_risk_ratios, **mr_trade_stats, "Max_Drawdown": mr_max_dd * 100.0}

print("=== Mean Reversion Scorecard ===")
mr_scorecard = generate_scorecard(mr_summary)
display(mr_scorecard)

### Inspect Top 5 Worst Mean Reversion Drawdown Events

In [ ]:
mr_dd_events = analyze_drawdowns(mr_portfolio["Portfolio_Value"])
display(mr_dd_events.head(5))

## 5. Generate Professional Visualizations

We call the plotting module to output the 20 charts for each strategy.

In [ ]:
figures_dir = "../reports/figures"

print("Plotting Momentum visualizations...")
plot_performance_dashboard(mom_portfolio, mom_trades, benchmark_df, output_dir=figures_dir, prefix="momentum")

print("Plotting Mean Reversion visualizations...")
plot_performance_dashboard(mr_portfolio, mr_trades, benchmark_df, output_dir=figures_dir, prefix="mean_reversion")

print("All 40 charts successfully plotted and saved to reports/figures!")

## 6. Export Performance Reports

We export scorecards and drawdown statistics to parquet and CSV files.

In [ ]:
# Export Momentum
mom_scorecard.to_csv("../data/processed/momentum_scorecard.csv", index=False)
mom_dd_events.to_csv("../data/processed/momentum_drawdowns.csv", index=False)

# Export Mean Reversion
mr_scorecard.to_csv("../data/processed/mean_reversion_scorecard.csv", index=False)
mr_dd_events.to_csv("../data/processed/mean_reversion_drawdowns.csv", index=False)

print("Performance summaries exported successfully.")